In [ ]:
import sys
import os
sys.path.append('../')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.verifier import verify_and_log

# 表示設定
pd.options.display.max_columns = None

# #01 岩石化学データ解析 (Pandas / Scikit-Learn)

## 0. 地球科学的背景: マグマの分化とMg#
マグマが冷えて結晶が分出する過程（結晶分化作用）では、Mgなどの元素が初期の結晶（かんらん石など）に取り込まれます。そのため、マグマの分化度を示す指標として **Mg# (Magnesium Number)** が広く使われます。

$$Mg\# = 100 \times \frac{MgO / 40.3}{(MgO / 40.3) + (FeO / 71.8)}$$

本実習では、大量の分析データからMg#を自動計算し、多変量解析（PCA）によって岩石種を分類するパイプラインを構築します。

## 1. ライブラリの基本操作 (写経)
まずは、Pandasによるデータの読み込みと、Scikit-Learnによる主成分分析の基本を学びましょう。

In [ ]:
# データの読み込み
df_sample = pd.read_csv('../data/raw_tabular/petrology.csv')
display(df_sample.head())

# 基本統計量の確認
print(df_sample.describe())

In [ ]:
from sklearn.decomposition import PCA

# データの正規化（ここでは簡易的に）
X = df_sample[['SiO2', 'MgO', 'FeO']].fillna(0)
pca = PCA(n_components=2)
components = pca.fit_transform(X)
print(f'寄与率: {pca.explained_variance_ratio_}')

## 🎯 Challenge: 自動解析パイプラインの構築
これまでに学んだ手法を組み合わせ、1つの関数として実装してください。

In [ ]:
def run_module_pipeline():
    # --- STEP 1: データの読み込み ---
    # data/raw_tabular/petrology.csv を読み込んでください
    df = pd.read_csv('../data/raw_tabular/petrology.csv')
    
    # --- STEP 2: 前処理 ---
    # 欠損値(NaN)を含む行を削除してください
    df = df.dropna()
    
    # --- STEP 3: 指標計算 ---
    # Mg# を計算し、新しい列 'Mg#' として追加してください
    # ヒント: 分母のゼロ割りに注意
    df['Mg#'] = 100 * (df['MgO']/40.3) / (df['MgO']/40.3 + df['FeO']/71.8)
    
    # --- STEP 4: 多変量解析 ---
    # SiO2, MgO, FeO を用いて PCA(n_components=2) を実行してください
    # 第1主成分が 0 より大きいか小さいかで 'Cluster' (0 or 1) を割り当ててください
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2)
    features = df[['SiO2', 'MgO', 'FeO']]
    pcs = pca.fit_transform(features)
    df['Cluster'] = (pcs[:, 0] > 0).astype(int)
    
    # --- STEP 5: 永続化 ---
    # 結果を SQLite 'data/petrology_processed.db' の 'analysis' テーブルに保存してください
    import sqlite3
    conn = sqlite3.connect('../data/petrology_processed.db')
    df.to_sql('analysis', conn, if_exists='replace', index=False)
    conn.close()
    
    return df

## ✅ 検証と記録
実行してレポートをクリップボードにコピーします。

In [ ]:
result = run_module_pipeline()
verify_and_log('M1', result)